In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

FINAL_PKL = "/Users/trinityrose/Desktop/acs_ssc_lasso_input.pkl"
OUT_PKL   = "/Users/trinityrose/Desktop/Replication Project/acs_ssc_predicted_second.pkl"

df = pd.read_pickle(FINAL_PKL)
print(f"Loaded: {df.shape}  |  Years: {sorted(df['year'].unique())}")

In [ ]:
# ── Fix missing sp_male ──────────────────────────────────────────
# build_data_V2.py omits sp_male from keep_cols.
if "sp_male" not in df.columns:
    df["sp_male"] = df["r_male"]
    print("Derived sp_male = r_male  (same-sex sample — sex is identical)")

# Confirm all required feature columns are present
needed = [
    "r_agegroup", "r_edugroup", "r_race", "r_occbroad", "r_degbroad", "r_male",
    "sp_agegroup", "sp_edugroup", "sp_race", "sp_occbroad", "sp_degbroad", "sp_male",
    "c_children", "statefip", "r_incearn", "sp_incearn"
]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Still missing columns: {missing}")
print("All columns present.")

In [ ]:
# ── Feature builder ──────────────────────────────────────────────
def build_features(person_df: pd.DataFrame) -> pd.DataFrame:
    """
    Expects columns already renamed to the canonical names:
      r_agegroup, r_edugroup, r_race, r_occbroad, r_degbroad,
      r_male, statefip, c_children
    Returns dummy-expanded + pairwise-interaction matrix.
    """
    cat_cols = ["r_agegroup", "r_edugroup", "r_race",
                "r_occbroad", "r_degbroad", "statefip"]

    base = pd.get_dummies(
        person_df[cat_cols + ["r_male", "c_children"]],
        columns=cat_cols,
        drop_first=True,
        dtype=float
    )

    # Pairwise interactions — keep only non-trivial (sum >= 10)
    base_cols = list(base.columns)
    interact_parts = []
    for i, c1 in enumerate(base_cols):
        for c2 in base_cols[i + 1:]:
            s = base[c1] * base[c2]
            if s.sum() >= 10:
                interact_parts.append(s.rename(f"{c1}_x_{c2}"))

    if interact_parts:
        X = pd.concat([base, pd.concat(interact_parts, axis=1)], axis=1)
    else:
        X = base

    return X.astype(float)


print("Feature builder defined.")

In [ ]:
def predict_income_for_role(df, train_mask, prefix="r", cv=10, max_iter=10_000):
    earn_col = f"{prefix}_incearn"

    X_full = build_features(df)
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_full)
    X_train = X_sc[train_mask.values]

    # Step 1: P(earn > 0)
    y1 = (df[earn_col] > 0).astype(float).values
    y1_train = y1[train_mask.values]

    m1 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m1.fit(X_train, y1_train)

    hat_pos_prob = m1.predict(X_sc)

    # Set threshold 
    target_share = y1.mean()                          
    threshold = np.percentile(hat_pos_prob, (1 - target_share) * 100)
    hat_pos = (hat_pos_prob >= threshold).astype(float)

    print(f"  [{prefix}] target share: {target_share:.3f}  "
      f"threshold: {threshold:.3f}  "
      f"achieved share: {hat_pos.mean():.3f}")

    # Step 2: earn in levels | earn > 0
    pos_train = train_mask.values & (df[earn_col].values > 0)
    y2_train = df.loc[pos_train, earn_col].values
    X2_train = X_sc[pos_train]

    m2 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m2.fit(X2_train, y2_train)

    hat_earn_lvl = np.maximum(m2.predict(X_sc), 0)
    hat_incearn = np.where(hat_pos == 1, hat_earn_lvl, 0.0)

    df = df.copy()
    df[f"hat_pos_{prefix}"]     = hat_pos
    df[f"hat_incearn_{prefix}"] = hat_incearn
    return df


In [ ]:
# Find the threshold 
y1_r = (df["r_incearn"] > 0).astype(float).values
train_mask = df["year"] == 2012

m1_temp = LassoCV(cv=10, max_iter=10_000, n_jobs=-1, random_state=42)
X_full = build_features(df)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_full)
m1_temp.fit(X_sc[train_mask.values], y1_r[train_mask.values])
hat_pos_prob_r = m1_temp.predict(X_sc)

paper_target_r  = 0.963   # from Table 2
paper_target_sp = 0.969

# Find threshold that achieves paper target
for t in np.arange(0.77, 0.95, 0.01):
    achieved = (hat_pos_prob_r >= t).mean()
    print(f"threshold={t:.2f}  achieved share={achieved:.3f}  target={paper_target_r:.3f}  diff={achieved-paper_target_r:+.3f}")

In [ ]:
y1_sp = (df["sp_incearn"] > 0).astype(float).values

m1_temp_sp = LassoCV(cv=10, max_iter=10_000, n_jobs=-1, random_state=42)
X_sc_sp = scaler.fit_transform(X_full)  # reuse X_full and scaler from above
m1_temp_sp.fit(X_sc_sp[train_mask.values], y1_sp[train_mask.values])
hat_pos_prob_sp = m1_temp_sp.predict(X_sc_sp)

paper_target_sp = 0.969

for t in np.arange(0.77, 0.95, 0.01):
    achieved = (hat_pos_prob_sp >= t).mean()
    print(f"threshold={t:.2f}  achieved share={achieved:.3f}  target={paper_target_sp:.3f}  diff={achieved-paper_target_sp:+.3f}")

In [ ]:
def predict_income_for_role(df, train_mask, prefix="r", cv=10, max_iter=10_000):
    earn_col = f"{prefix}_incearn"

    X_full = build_features(df)
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_full)
    X_train = X_sc[train_mask.values]

    y1 = (df[earn_col] > 0).astype(float).values
    y1_train = y1[train_mask.values]

    m1 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m1.fit(X_train, y1_train)

    hat_pos_prob = m1.predict(X_sc)

    # Thresholds
    threshold = 0.78 if prefix == "r" else 0.77
    hat_pos = (hat_pos_prob >= threshold).astype(float)

    print(f"  [{prefix}] threshold={threshold}  "
          f"achieved={hat_pos.mean():.3f}  "
          f"paper target={'0.963' if prefix=='r' else '0.969'}")

    pos_train = train_mask.values & (df[earn_col].values > 0)
    y2_train = df.loc[pos_train, earn_col].values
    X2_train = X_sc[pos_train]

    m2 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m2.fit(X2_train, y2_train)

    hat_earn_lvl = np.maximum(m2.predict(X_sc), 0)
    hat_incearn = np.where(hat_pos == 1, hat_earn_lvl, 0.0)

    df = df.copy()
    df[f"hat_pos_{prefix}"]     = hat_pos
    df[f"hat_incearn_{prefix}"] = hat_incearn
    return df


In [ ]:
# ── Run — Respondent ────────────────────────────────────────────
train_mask = df["year"] == 2012
print(f"Training sample (2012): N={train_mask.sum():,}")

df = predict_income_for_role(df, train_mask, prefix="r")

In [ ]:
# ── Run — Spouse / partner ──────────────────────────────────────
df = predict_income_for_role(df, train_mask, prefix="sp")

In [ ]:
# ── Couple-level aggregates ──────────────────────────────────────
df["hat_c_incearn"] = df["hat_incearn_r"] + df["hat_incearn_sp"]

# Primary earner share = max(r, sp) / total  (matches paper's Table 2)
total = df["hat_c_incearn"].replace(0, np.nan)
df["hat_c_split"] = (
    np.maximum(df["hat_incearn_r"], df["hat_incearn_sp"]) / total
).fillna(0.5)

# Store the 5-polynomial terms
for i in range(1, 6):
    df[f"hat_c_incearn{i}"] = (df["hat_c_incearn"] / 10_000) ** i
    df[f"hat_c_split{i}"]   = df["hat_c_split"] ** i

print("Couple-level columns added.")

In [ ]:
# ── Diagnostics vs Table 2 ───────────────────────────────────────
print("="*60)
print("DIAGNOSTICS vs. Table 2")
print("="*60)

print("\n--- Share positive earnings ---")
print(f"  Respondent  reported={(df['r_incearn']>0).mean():.3f}  "
      f"predicted={df['hat_pos_r'].mean():.3f}  "
      f"(paper predicted: 0.963 mar / 0.969 coh)")
print(f"  Spouse      reported={(df['sp_incearn']>0).mean():.3f}  "
      f"predicted={df['hat_pos_sp'].mean():.3f}")

print("\n--- Couple predicted total earnings ---")
for label, mask in [("Married", df["sscouple_mar"]),
                    ("Cohabiting", df["sscouple_coh"])]:
    sub = df[mask]
    print(f"  {label}: {sub['hat_c_incearn'].mean():>10,.0f}  "
          f"(paper: mar≈110,729  coh≈102,953)")

print("\n--- Couple predicted earnings split ---")
for label, mask in [("Married", df["sscouple_mar"]),
                    ("Cohabiting", df["sscouple_coh"])]:
    sub = df[mask]
    pos = sub["hat_c_incearn"] > 0
    print(f"  {label}: {sub.loc[pos,'hat_c_split'].mean():.3f}  "
          f"(paper: mar≈0.648  coh≈0.641)")

print("\n--- Correlation reported vs predicted (positive earners) ---")
for pfx, label in [("r", "Respondent"), ("sp", "Spouse")]:
    both = (df[f"{pfx}_incearn"] > 0) & (df[f"hat_incearn_{pfx}"] > 0)
    r = np.corrcoef(df.loc[both, f"{pfx}_incearn"],
                    df.loc[both, f"hat_incearn_{pfx}"])[0, 1]
    print(f"  {label}: r={r:.3f}")

In [ ]:
# ── Save ─────────────────────────────────────────────────────────
df.to_pickle(OUT_PKL)
print(f"Saved to {OUT_PKL}")